# FacePred: Colab Free Real Training

Run Sections 1-3 on a CPU runtime. After the full cache completes, switch to a GPU runtime and run Sections 1, 4, 5, and 6.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/AcastaPaloma/facepred.git'
DRIVE_ROOT = '/content/drive/MyDrive/facepred'
CACHE_NAME = 'meld_audio_causal_v1'
CAMPAIGN_NAME = 'audio_campaign_v1'

## 1. Setup or recovery setup

In [ ]:
!cd /content && (git clone $REPO_URL facepred || true)
%cd /content/facepred
!git pull
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip install -e .
!pip install numpy==1.26.4 pandas scipy pyyaml huggingface_hub pytest ruff
!python -m ruff check .
!python -m pytest -q

## 2. Download and locally extract real MELD

This stores the approximately 10.9 GB archive in Drive, recursively extracts it onto temporary Colab disk, and fetches the official annotation CSVs. Continue only when the output reports archive_gb near 10.9, three CSVs, and a nonzero media_files count.

In [ ]:
!python scripts/stage_meld_colab.py \
  --archive "$DRIVE_ROOT/data/MELD.Raw.tar.gz" \
  --local-archive /content/MELD.Raw.tar.gz \
  --extract-dir /content/facepred_data

## 3. Build resumable real-audio cache

Run the smoke cell first. Then run the full cell. If the runtime dies, repeat Sections 1-3; completed dialogues are skipped.

In [ ]:
!python scripts/prepare_meld_audio_cache.py \
  --data-root /content/facepred_data \
  --output-dir "$DRIVE_ROOT/cache/meld_audio_causal_smoke" \
  --max-dialogues 2

In [ ]:
!python scripts/prepare_meld_audio_cache.py \
  --data-root /content/facepred_data \
  --output-dir "$DRIVE_ROOT/cache/$CACHE_NAME"

## 4. GPU runtime: copy finished cache locally

In [ ]:
import torch

assert torch.cuda.is_available(), 'Switch the Colab runtime to GPU before training.'
print(torch.cuda.get_device_name(0))
!rm -rf /content/facepred_cache
!mkdir -p /content/facepred_cache
!rsync -a --exclude '.progress/' "$DRIVE_ROOT/cache/$CACHE_NAME/" /content/facepred_cache/

## 5. Tune and continue the winner to 75 epochs

Rerun this same cell after any disconnect. It resumes from Drive checkpoints.

In [ ]:
!python scripts/tune_and_train_colab.py \
  --cache-dir /content/facepred_cache \
  --output-root "$DRIVE_ROOT/runs/$CAMPAIGN_NAME" \
  --device cuda \
  --batch-size 32 \
  --stage1-epochs 5 \
  --stage2-epochs 15 \
  --max-epochs 75 \
  --save-every-steps 100 \
  --early-stopping-patience 12

## 6. One final test evaluation

In [ ]:
!python scripts/evaluate_selected_run.py \
  --selection "$DRIVE_ROOT/runs/$CAMPAIGN_NAME/selection.json" \
  --cache-dir /content/facepred_cache \
  --split test \
  --device cuda